In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from constrerl.erl_schema import (
    entity_labels,
    relations,
)
from constrerl.annotator import Annotator, AnnotationTypes, article_to_sentences
from constrerl.annotation_model import (
    AnnotatedArticle,
    load_collection,
    Entity,
    AnnotatedArticle,
)
from constrerl.sentences import BERTAnnotator, SpacyAnnotator

In [3]:
dev_articles=load_collection("Dev")
train_articles=load_collection("Train")

In [4]:
from constrerl.utils import prepare_for_eval
from constrerl.eval.evaluate import eval_submission_NED

In [5]:
from constrerl.sentences import SentenceAnnotator


ned_annotators: dict[str, SentenceAnnotator] = {
    "BERT": BERTAnnotator(),
    "Spacy": SpacyAnnotator(),
}


In [6]:
def tuple_to_scores(tup):
    # precision, recall, f1, micro_precision, micro_recall, micro_f1
    return {
        "precision": tup[0],
        "recall": tup[1],
        "f1": tup[2],
        "micro_precision": tup[3],
        "micro_recall": tup[4],
        "micro_f1": tup[5],
    }



In [7]:
import tqdm

from constrerl.sentences import annotated_sentences_to_article, Sentence


scores_list = []
for name, annotator in ned_annotators.items():
    print(f"Evaluating {name} annotator...")
    predictions = {}
    converted_articles = {}
    for id, article in tqdm.tqdm(dev_articles.items(), desc="Processing articles"):
        article_cp = AnnotatedArticle(**article.__dict__)
        for ent in article_cp.entities:
            ent.label = "NOUN"
        sentences = article_to_sentences(article.metadata)

        annotated_sentences = []
        for sentence in sentences:
            sentence_cp = Sentence(**sentence.__dict__)
            sentence_predictions = annotator.extract_noun_phrases(sentence.text)
            sentence_cp.entities = [
                Entity(
                    label="NOUN",
                    start_idx=pred.start_idx,
                    end_idx=pred.end_idx,
                    text_span=pred.text,
                    location="title" if sentence.title else "abstract",
                )
                for span, pred in sentence_predictions.items()
            ]
            annotated_sentences.append(sentence_cp)
        predictions[id] = annotated_sentences_to_article(
            annotated_sentences, article.metadata
        )
    res = eval_submission_NED(
        prepare_for_eval(predictions), prepare_for_eval(dev_articles)
    )
    scores = tuple_to_scores(res)
    scores["type"] = name
    scores_list.append(scores)

Evaluating BERT annotator...


Processing articles:   0%|          | 0/80 [00:00<?, ?it/s]

Processing articles: 100%|██████████| 80/80 [00:17<00:00,  4.55it/s]


Evaluating Spacy annotator...


Processing articles: 100%|██████████| 80/80 [00:21<00:00,  3.64it/s]


In [8]:
sentence_predictions

{'it': AnnotationSpan(start_idx=9, end_idx=10, text='it'),
 'many more chronic, non-communicable, inflammatory diseases': AnnotationSpan(start_idx=23, end_idx=80, text='many more chronic, non-communicable, inflammatory diseases'),
 'bactericidal antibiotics': AnnotationSpan(start_idx=171, end_idx=194, text='bactericidal antibiotics'),
 'vaccines': AnnotationSpan(start_idx=199, end_idx=206, text='vaccines'),
 'microbial component': AnnotationSpan(start_idx=93, end_idx=111, text='microbial component')}

In [9]:
import pandas as pd
df_scores = pd.DataFrame(scores_list)
df_scores

,precision,recall,f1,micro_precision,micro_recall,micro_f1,type
0,0.685606,0.646172,0.665305,0.685606,0.646172,0.665305,BERT
1,0.342194,0.801666,0.479649,0.342194,0.801666,0.479649,Spacy
